# YOLO Object Detection
We will train our own weights for the YOLOv7 model. We have one class we want to detect, snowcones. There's two datasets, one with Lidar images and one with RGB images. We can train our model to either handle both or one model for each of them.

## Importing packages

In [29]:
import os
import re
import time
import csv
import glob
import pandas as pd

In [ ]:
!pip install --user -r requirements.txt

In [ ]:
!wget https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7_training.pt

In [ ]:
# Create local links inside your project for RGB
!mkdir -p data/rgb/images/train data/rgb/images/valid data/rgb/images/test
!mkdir -p data/rgb/labels/train data/rgb/labels/valid data/rgb/labels/test

!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/train/* data/rgb/images/train/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/valid/* data/rgb/images/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/images/test/* data/rgb/images/test/

!ln -s /datasets/tdt4265/ad/open/Poles/rgb/labels/train/* data/rgb/labels/train/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/labels/valid/* data/rgb/labels/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/rgb/labels/test/* data/rgb/labels/test/ #There are no labels for test dataset

# Similarly for LIDAR
!mkdir -p data/lidar/combined_color/train data/lidar/combined_color/valid data/lidar/combined_color/test
!mkdir -p data/lidar/labels/train data/lidar/labels/valid data/lidar/labels/test

!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/train/* data/lidar/combined_color/train/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/valid/* data/lidar/combined_color/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/test/* data/lidar/combined_color/test/

!ln -s /datasets/tdt4265/ad/open/Poles/lidar/labels/train/* data/lidar/labels/train/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/labels/valid/* data/lidar/labels/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/labels/test/* data/lidar/labels/test/ #There are no labels for test dataset

In [50]:
!mkdir -p data/lidar/images/train data/lidar/images/valid data/lidar/images/test

!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/train/* data/lidar/images/train/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/valid/* data/lidar/images/valid/
!ln -s /datasets/tdt4265/ad/open/Poles/lidar/combined_color/test/* data/lidar/images/test/

In [ ]:
!mkdir -p data/combined/images/train data/combined/images/valid
!mkdir -p data/combined/labels/train data/combined/labels/valid

# ---- TRAINING DATA ----
# Link RGB images and labels
!ln -s $(realpath data/rgb/images/train/*) data/combined/images/train/
!ln -s $(realpath data/rgb/labels/train/*) data/combined/labels/train/

# Link LIDAR images and labels
!ln -s $(realpath data/lidar/combined_color/train/*) data/combined/images/train/
!ln -s $(realpath data/lidar/labels/train/*) data/combined/labels/train/

# ---- VALIDATION DATA ----
# Link RGB images and labels
!ln -s $(realpath data/rgb/images/valid/*) data/combined/images/valid/
!ln -s $(realpath data/rgb/labels/valid/*) data/combined/labels/valid/

# Link LIDAR images and labels
!ln -s $(realpath data/lidar/combined_color/valid/*) data/combined/images/valid/
!ln -s $(realpath data/lidar/labels/valid/*) data/combined/labels/valid/



## Training YOLO model
Finding latest weights

In [10]:

def latestWeights():
    # Get all exp folders in runs/train
    exp_dirs = [d for d in os.listdir("runs/train") if re.match(r"exp\d*$", d)]

    # Sort by numeric suffix (exp, exp2, exp3, ...)
    exp_dirs.sort(key=lambda x: int(x[3:]) if x != "exp" else 0)

    # Get the latest one
    latest_exp = exp_dirs[-1]

    best_path = f"runs/train/{latest_exp}/weights/best.pt"
    if not os.path.exists(best_path):
        print(f"Warning: best.pt not found in {latest_exp}. Using last.pt instead.")
        best_path = f"runs/train/{latest_exp}/weights/last.pt"
    print("Using weights from:", best_path)
    return best_path

In [5]:
def get_latest_run_weights(project_dir='runs/train_lidar'):
    """Finds the best.pt file from the latest run in a project directory."""
    last_run = None
    last_exp_num = -1

    if not os.path.exists(project_dir):
        print(f"Error: Project directory '{project_dir}' not found.")
        return None

    # Find the experiment directory with the highest number (e.g., exp, exp2, exp3 -> finds exp3)
    for d in os.listdir(project_dir):
        if d.startswith('exp'):
            try:
                # Extract number (exp -> 0, expN -> N)
                num = int(d[3:]) if len(d) > 3 else 0
                if num > last_exp_num:
                    last_exp_num = num
                    last_run = os.path.join(project_dir, d)
            except ValueError:
                continue # Ignore directories not matching exp<number> pattern

    if last_run is None:
         # Fallback if only 'exp' exists without numbers
         if os.path.exists(os.path.join(project_dir, 'exp')):
             last_run = os.path.join(project_dir, 'exp')
         else:
             print(f"Error: No experiment runs found in '{project_dir}'.")
             return None

    weights_dir = os.path.join(last_run, 'weights')
    best_weights = os.path.join(weights_dir, 'best.pt')
    last_weights = os.path.join(weights_dir, 'last.pt')

    if os.path.exists(best_weights):
        print(f"Using best weights from: {best_weights}")
        return best_weights
    elif os.path.exists(last_weights):
        print(f"Warning: best.pt not found in {last_run}. Using last.pt instead: {last_weights}")
        return last_weights
    else:
        print(f"Error: No best.pt or last.pt found in {weights_dir}.")
        return None

Training the data, either on the latest weights or the initial COCO weights

In [39]:
# ---- COMBINED TRAINING ---- #
initial_weights = "yolov7_training.pt"
batch_number = 16
epochs_number = 400
img_size = 1000
print("--- Starting Combined Training ---")
start_time_lidar = time.time()
!python train.py --batch {batch_number} \
                 --epochs {epochs_number} \
                 --data combined.yaml \
                 --weights {initial_weights} \
                 --device 0 \
                 --img-size {img_size} \
                 --rect \
                 --project runs/train_combined --name exp # Save to separate project

end_time_lidar = time.time()
elapsed_time_lidar = end_time_lidar - start_time_lidar
print(f"LIDAR Training time: {elapsed_time_lidar:.2f} seconds")

--- Starting Combined Training ---
2025-05-01 16:05:42.300971: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-01 16:05:42.325069: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-01 16:05:42.741662: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
YOLOR 🚀 44c67b0 torch 2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4090, 24095.5625MB)

Namespace(weights='yolov7_training.pt', cfg='', data='combined.yaml', hyp='data/hyp.scratch.p5.yaml', epochs=400, batch_s

In [12]:
# --- Configuration ---
initial_weights = "yolov7_training.pt"
batch_number_lidar = 32
epochs_number_lidar = 150
img_size_lidar = 1408

batch_number_rgb = 16 # Can be different if needed
epochs_number_rgb = 300
img_size_rgb = 960


In [ ]:

# --- Run 1: LIDAR Training ---
print("--- Starting LIDAR Training ---")
start_time_lidar = time.time()
!python train.py --batch {batch_number_lidar} \
                 --epochs {epochs_number_lidar} \
                 --data lidar.yaml \
                 --weights {initial_weights} \
                 --device 0 \
                 --img-size {img_size_lidar} \
                 --rect \
                 --project runs/train_lidar --name exp # Save to separate project

end_time_lidar = time.time()
elapsed_time_lidar = end_time_lidar - start_time_lidar
print(f"LIDAR Training time: {elapsed_time_lidar:.2f} seconds")

In [35]:
###training RGB####

# --- Configuration ---
initial_weights = "yolov7_training.pt"
custom_hyp = "/work/chrsjoha/SnowConeDetection/yolo/yolov7-main/data/hyp.scratch.custom.lidar.yaml"
custom_hyp = "hyp.scratch.custom.lidar.yaml"
batch_number_lidar = 16
epochs_number_lidar = 150
img_size_lidar = 1280

batch_number_rgb = 16 # Can be different if needed
epochs_number_rgb = 30
img_size_rgb = 960

# --- Run 2: RGB Training ---
print("\n--- Starting RGB Training ---")
start_time_rgb = time.time()
!python train.py --batch {batch_number_rgb} \
                 --epochs {epochs_number_rgb} \
                 --data rgb.yaml \
                 --weights {initial_weights} \
                 --device 0 \
                 --img-size {img_size_rgb} \
                 --project runs/train_rgb --name exp 
                 #--hyp {custom_hyp}

end_time_rgb = time.time()
elapsed_time_rgb = end_time_rgb - start_time_rgb
print(f"RGB Training time: {elapsed_time_rgb:.2f} seconds")


--- Starting RGB Training ---
2025-05-01 15:55:42.333571: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-01 15:55:42.357628: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-05-01 15:55:42.764672: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
YOLOR 🚀 44c67b0 torch 2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4090, 24095.5625MB)

Namespace(weights='yolov7_training.pt', cfg='', data='rgb.yaml', hyp='data/hyp.scratch.p5.yaml', epochs=30, batch_size=16, im

In [ ]:
path_base_weights = "yolov7_training.pt"
base_training = False 
if(not base_training)
    path_latest_weights = latestWeights()
#False -> Use COCO weights - Initilization
#True  -> Use latest weights - Continue training our model

batch_number = 16
epochs_number = 55
start_time = time.time()
if(not base_training):
    !python train.py --batch {batch_number} --epochs {epochs_number} --data lidar.yaml --weights {path_latest_weights} --device 0
    !python train.py --batch {batch_number} --epochs {epochs_number} --data rgb.yaml --weights {path_latest_weights} --device 0 
else:
    !python train.py --batch {batch_number} --epochs {epochs_number} --data lidar.yaml --weights {path_base_weights} --device 0
    !python train.py --batch {batch_number} --epochs {epochs_number} --data rgb.yaml --weights {path_base_weights} --device 0
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Training time: {elapsed_time:.2f} seconds") 
# Save training data time to csv file
with open("training_times.csv", "a", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([time.strftime("%Y-%m-%d %H:%M:%S"), elapsed_time, batch_number, epochs_number, weights_path])

## Testing weights on the test-dataset
First we find the latest weights by looking for the last exp in the 'runs/train' folder

In [43]:
#best_path = latestWeights()
lidar_weights_path = get_latest_run_weights(project_dir='runs/train_lidar') # Assumes you used --project runs/train_lidar
rgb_weights_path = get_latest_run_weights(project_dir='runs/train_rgb') # Assumes you used --project runs/train_lidar
lidar_weights_path = get_latest_run_weights(project_dir='runs/train_combined') # Assumes you used --project runs/train_lidar
rgb_weights_path = get_latest_run_weights(project_dir='runs/train_combined') # Assumes you used --project runs/train_lidar
combined_weights_path = get_latest_run_weights(project_dir='runs/train_combined')


Error: Project directory 'runs/train_lidar' not found.
Using best weights from: runs/train_rgb/exp5/weights/best.pt
Using best weights from: runs/train_combined/exp3/weights/best.pt
Using best weights from: runs/train_combined/exp3/weights/best.pt
Using best weights from: runs/train_combined/exp3/weights/best.pt


Then we use these weights to do a detection test on the test dataset

lidar test

In [51]:
if lidar_weights_path:
    img_size_detect = 1024 # Use the same img-size as training
    conf_threshold = 0.05 # Confidence threshold - adjust as needed
    iou_threshold = 0.45  # IoU threshold for NMS - adjust as needed
    test_images_source = 'data/lidar/images/test/' # Path to your LIDAR test images
    output_project = 'runs/detect_lidar'
    output_name = 'exp_test_results'

    print(f"\n--- Running Detection on LIDAR Test Set ---")
    # Construct the command
    detect_command = f"python detect.py --weights {lidar_weights_path} \
                                        --source {test_images_source} \
                                        --img-size {img_size_detect} \
                                        --conf-thres {conf_threshold} \
                                        --iou-thres {iou_threshold} \
                                        --save-txt \
                                        --save-conf \
                                        --project {output_project} \
                                        --name {output_name} \
                                        --nosave \
                                        --device 0" # Optional: add --nosave if you don't need image outputs

    print(f"Executing: {detect_command}")
    # Execute the command
    !{detect_command}

    print(f"\nDetection complete. Results saved in '{os.path.join(output_project, output_name, 'labels')}'")
    # You can verify the output format:
    print("\nExample output file content (first 5 lines of a sample file):")
    try:
      output_label_dir = os.path.join(output_project, output_name, 'labels')
      sample_file = glob.glob(os.path.join(output_label_dir, '*.txt'))[0]
      with open(sample_file, 'r') as f:
          for i, line in enumerate(f):
              if i >= 5: break
              print(line.strip())
    except Exception as e:
        print(f"Could not read sample output file: {e}")

else:
    print("Could not find LIDAR weights to run detection.")


--- Running Detection on LIDAR Test Set ---
Executing: python detect.py --weights runs/train_combined/exp3/weights/best.pt                                         --source data/lidar/images/test/                                         --img-size 1024                                         --conf-thres 0.05                                         --iou-thres 0.45                                         --save-txt                                         --save-conf                                         --project runs/detect_lidar                                         --name exp_test_results                                         --nosave                                         --device 0
Namespace(weights=['runs/train_combined/exp3/weights/best.pt'], source='data/lidar/images/test/', img_size=1024, conf_thres=0.05, iou_thres=0.45, device='0', view_img=False, save_txt=True, save_conf=True, nosave=True, classes=None, agnostic_nms=False, augment=False, update=False, project='runs/de

In [47]:
#rgb_weights_path = 'runs/train_rgb/exp3/weights/best.pt'
if rgb_weights_path:
    img_size_detect_rgb = 1024  # Image size used for RGB training/detection
    conf_threshold = 0.05 # Confidence threshold - adjust as needed
    iou_threshold = 0.45  # IoU threshold for NMS - adjust as needed
    test_images_source_rgb = 'data/rgb/images/test/' # Path to your RGB test images
    output_project_rgb = 'runs/detect_rgb'
    output_name_rgb = 'exp_test_results'

    print(f"\n--- Running Detection on RGB Test Set ---")
    # Construct the command
    detect_command_rgb = f"python detect.py --weights {rgb_weights_path} \
                                            --source {test_images_source_rgb} \
                                            --img-size {img_size_detect_rgb} \
                                            --conf-thres {conf_threshold} \
                                            --iou-thres {iou_threshold} \
                                            --save-txt \
                                            --save-conf \
                                            --project {output_project_rgb} \
                                            --name {output_name_rgb} \
                                            --nosave \
                                            --device 0" # Optional: add --nosave if you don't need image outputs

    print(f"Executing: {detect_command_rgb}")
    # Execute the command
    !{detect_command_rgb}

    print(f"\nDetection complete. Results saved in '{os.path.join(output_project_rgb, output_name_rgb, 'labels')}'")
    # You can verify the output format:
    print("\nExample output file content (first 5 lines of a sample file):")
    try:
      output_label_dir_rgb = os.path.join(output_project_rgb, output_name_rgb, 'labels')
      sample_file_rgb = glob.glob(os.path.join(output_label_dir_rgb, '*.txt'))[0]
      with open(sample_file_rgb, 'r') as f:
          for i, line in enumerate(f):
              if i >= 5: break
              print(line.strip())
    except Exception as e:
        print(f"Could not read sample output file: {e}")

else:
    print("Could not find RGB weights to run detection.")


--- Running Detection on RGB Test Set ---
Executing: python detect.py --weights runs/train_combined/exp3/weights/best.pt                                             --source data/rgb/images/test/                                             --img-size 1024                                             --conf-thres 0.05                                             --iou-thres 0.45                                             --save-txt                                             --save-conf                                             --project runs/detect_rgb                                             --name exp_test_results                                             --nosave                                             --device 0
Namespace(weights=['runs/train_combined/exp3/weights/best.pt'], source='data/rgb/images/test/', img_size=1024, conf_thres=0.05, iou_thres=0.45, device='0', view_img=False, save_txt=True, save_conf=True, nosave=True, classes=None, agnostic_nms=False, augment=False

In [ ]:
manual_detection = False #Change this to True if you want to select your own weights, make sure it has a "best.pt"
weight_number = 16 # expXX

if(not manual_detection):
    !python detect.py --weights {best_path} --conf 0.1 --source data/lidar/images/test
    !python detect.py --weights {best_path} --conf 0.1 --source data/rgb/images/test
else:
    !python detect.py --weights runs/train/exp{weights_number}/weights/best.pt --conf 0.1 --source data/lidar/combined_color/test
    !python detect.py --weights runs/train/exp{weights_number}/weights/best.pt --conf 0.1 --source data/rgb/images/test

## Plotting Results

In [41]:
power_path = '/work/chrsjoha/SnowConeDetection/power_log_combined.csv'

df = pd.read_csv(power_path, header=None, names=['timestamp', 'power'])

df['power'] = df['power'].str.replace(' W', '', regex=False).astype(float)

average_power = df['power'].mean()

print(f'Average Power: {average_power:.2f} W')


Average Power: 250.12 W
